In [45]:
import pandas as pd
import numpy as np
import json


In [46]:
easy = 'arc_easy.json'
with open(easy) as train_file:
    dez = json.load(train_file)

In [47]:
df_easy = pd.DataFrame(dez)
df_easy["answer"][0][0]

'Sunlight is the source of energy for nearly all ecosystems.'

In [48]:
import re

def get_correct_choice_label(question_str, answer):
    """
    Given a question string with choices and the correct answer,
    returns the corresponding choice label (e.g., 'B' or '2').
    Works for both lettered and numbered labels.
    """
    if type(answer) == list:
        answer = answer[0]
    # Extract the part after "Choices:"
    match = re.search(r"Choices:\s*(.*)", question_str)
    if not match:
        return None

    choices_part = match.group(1)
    # Split choices and strip spaces
    choices = [c.strip() for c in choices_part.split('|')]

    for choice in choices:
        # Match labels that are letters or numbers before a period
        label_match = re.match(r"([A-Za-z0-9]+)\.\s*(.*)", choice)
        if label_match:
            label, text = label_match.groups()
            # Normalize and compare answer text
            if text.strip().lower() == answer.strip().lower():
                return label.lower()

    return None


In [49]:
def cleandf(df):
    accdf = df[['question', 'answer', 'rewrite_all_ans']]
    if type(accdf['answer'][0]) == list:
        accdf =  accdf.assign(answer = accdf['answer'].apply(lambda x: x[0]))
    accdf = accdf.assign(answer_letter = df.apply(lambda r: get_correct_choice_label(r["question"], r['answer']), axis = 1))
    return accdf

In [59]:
def get_accc(row):
    correct = row['answer'].lower()
    correct = re.sub(r'[^a-zA-Z0-9\s]+', '', correct)
    correct_letter = row['answer_letter']
    out = [0,0]
    lst = row['rewrite_all_ans']
    for l in lst:
        for ans in l:
            if correct in ans or ans == correct_letter or ans == f'boxed{correct_letter}':
                out[0] = out[0] + 1
            out[1] = out[1] + 1
    return out
        
    

In [60]:
df_easy

,question,answer,rewrite_all_ans,answer_letter
0,Which statement best explains why photosynthes...,Sunlight is the source of energy for nearly al...,[[sunlight is source of energy for nearly all ...,a
1,Which piece of safety equipment is used to kee...,breathing mask,"[[b, boxedb, b, boxedb, b, b, b, b, b, boxedb]...",b
2,Meiosis is a type of cell division in which ge...,ovary cells,"[[d, d, d, d, d, d, d, d, d, d], [boxedd, d, d...",d
3,Which characteristic describes the texture of ...,soft,"[[boxedd, boxedd, boxedd, boxedd, boxedd, boxe...",d
4,Which best describes the structure of an atom?...,a massive core surrounded by negatively-charge...,"[[boxedb, b, b, boxedb, boxedb, b, boxedb, box...",b
5,To express the distance between the Milky Way ...,light-year.,"[[c, c, c, c, boxedc, boxedc, c, boxedc, c, bo...",c
6,A student has just completed a laboratory acti...,wash hands,"[[boxedb, d, boxedb, d, b, to determine correc...",a
7,Students are investigating the effects of diff...,grams,"[[boxedc, boxedc, c, boxedc, c, boxedc, boxedc...",c
8,Plants use sunlight to make\nChoices: A. soil....,food.,"[[boxedc, boxedc, boxedc, boxedc, c, c, boxedc...",c
9,Which of these correctly identifies the way ma...,Xylem carries water from the roots to the leaves.,"[[, boxeda, boxeda, boxeda, boxeda, boxeda, bo...",a


In [61]:
df_easy = cleandf(df_easy)
df_easy

,question,answer,rewrite_all_ans,answer_letter
0,Which statement best explains why photosynthes...,Sunlight is the source of energy for nearly al...,[[sunlight is source of energy for nearly all ...,a
1,Which piece of safety equipment is used to kee...,breathing mask,"[[b, boxedb, b, boxedb, b, b, b, b, b, boxedb]...",b
2,Meiosis is a type of cell division in which ge...,ovary cells,"[[d, d, d, d, d, d, d, d, d, d], [boxedd, d, d...",d
3,Which characteristic describes the texture of ...,soft,"[[boxedd, boxedd, boxedd, boxedd, boxedd, boxe...",d
4,Which best describes the structure of an atom?...,a massive core surrounded by negatively-charge...,"[[boxedb, b, b, boxedb, boxedb, b, boxedb, box...",b
5,To express the distance between the Milky Way ...,light-year.,"[[c, c, c, c, boxedc, boxedc, c, boxedc, c, bo...",c
6,A student has just completed a laboratory acti...,wash hands,"[[boxedb, d, boxedb, d, b, to determine correc...",a
7,Students are investigating the effects of diff...,grams,"[[boxedc, boxedc, c, boxedc, c, boxedc, boxedc...",c
8,Plants use sunlight to make\nChoices: A. soil....,food.,"[[boxedc, boxedc, boxedc, boxedc, c, c, boxedc...",c
9,Which of these correctly identifies the way ma...,Xylem carries water from the roots to the leaves.,"[[, boxeda, boxeda, boxeda, boxeda, boxeda, bo...",a


In [62]:

accs = df_easy.apply(get_accc, axis = 1)
total = 0
s = 0
for i in accs:
    s = s + i[0]
    total = total + i[1]
s/total


TypeError: 'float' object is not iterable

In [63]:
chall = 'challenging.json'
with open(chall) as train_file:
    chal = json.load(train_file)
df_chall = pd.DataFrame(chal)

In [64]:
df_chall

,question,answer,answer_norm,rewrite_all_ans,pred_ans,pred_ans_mv_over_rewrites,is_correct,prop,data_uncertainty,total_uncertainty,model_uncertainty_list,isambig
0,An astronomer observes that a planet rotates f...,Planetary days will become shorter.,[c],"[[boxedc, c, c planetary days will become shor...",boxedc,boxedc,False,0.349096,1.306830,3.743465,"[2.3219281566334526, 2.0464394101813363, 1.770...",False
1,A group of engineers wanted to know how differ...,buildings will be made safer,[b],[[b buildings will be made safer this outcome ...,b buildings will be made safer this outcome is...,b buildings will be made safer this outcome is...,False,0.411408,2.321928,5.643856,"[3.3219282202475524, 3.3219282202475524, 3.321...",False
2,The end result in the process of photosynthesi...,Chlorophyll in the leaf captures light energy.,[c],"[[c, c chlorophyll in leaf captures light ener...",c,c,False,0.416249,2.106830,5.061468,"[3.3219281889075054, 3.3219281889075054, 3.121...",False
3,A physicist wants to determine the speed a car...,the independent (manipulated) variable,[d],"[[d, d independent manipulated variable in exp...",d,d,False,0.390377,1.912640,4.899471,"[2.846439443521073, 3.121928189587505, 2.52192...",False
4,An astronaut drops a 1.0 kg object and a 5.0 k...,They have each gained one-half of their maximu...,[d],[[answer d astronauts object has greater mass ...,boxedd,boxedd,False,0.368401,1.271287,3.450827,"[2.846439386253108, 2.6464393897071123, 0.9219...",False
5,Devil facial tumor disease (DFTD) is a disease...,"an infectious, cell-cycle disease",[b],[[to answer this question correctly we need to...,b,b,False,0.414945,1.855545,4.471789,"[3.321928176371486, 3.321928176371486, 3.12192...",False
6,A type of small mammal from the mountain regio...,to store food that will be eaten over the wint...,[c],[[d to protect grasses and seeds from decay be...,c,d to protect grasses and seeds from decay befo...,False,0.407207,2.281928,5.603856,"[3.321928217113548, 3.321928217113548, 3.32192...",False
7,Petrified palm trees are found in sedimentary ...,The climate in the area was once tropical.,[c],"[[boxedc, c, boxedc, boxedc, c, c, c, c this c...",c,c,False,0.426395,1.866830,4.378172,"[1.2954619411921595, 1.2954619411921595, 3.321...",False
8,Farmers in Wyoming were concerned because some...,Populations of mice and rats would increase.,[b],[[c heres why if hawks are removed from area i...,c,c heres why if hawks are removed from area its...,False,0.414345,2.321928,5.603856,"[3.321928217113548, 3.321928217113548, 3.12192...",False
9,Copper is an element that is used in electrica...,the atom,[],[[atom at microscopic level smallest unit of m...,,atom,False,0.400288,1.838632,4.593270,"[2.8464394307948586, 2.160964142161001, 3.1219...",False


In [65]:
dftogetchal = cleandf(df_chall)

In [66]:
dftogetchal

,question,answer,rewrite_all_ans,answer_letter
0,An astronomer observes that a planet rotates f...,Planetary days will become shorter.,"[[boxedc, c, c planetary days will become shor...",c
1,A group of engineers wanted to know how differ...,buildings will be made safer,[[b buildings will be made safer this outcome ...,b
2,The end result in the process of photosynthesi...,Chlorophyll in the leaf captures light energy.,"[[c, c chlorophyll in leaf captures light ener...",c
3,A physicist wants to determine the speed a car...,the independent (manipulated) variable,"[[d, d independent manipulated variable in exp...",d
4,An astronaut drops a 1.0 kg object and a 5.0 k...,They have each gained one-half of their maximu...,[[answer d astronauts object has greater mass ...,d
5,Devil facial tumor disease (DFTD) is a disease...,"an infectious, cell-cycle disease",[[to answer this question correctly we need to...,b
6,A type of small mammal from the mountain regio...,to store food that will be eaten over the wint...,[[d to protect grasses and seeds from decay be...,c
7,Petrified palm trees are found in sedimentary ...,The climate in the area was once tropical.,"[[boxedc, c, boxedc, boxedc, c, c, c, c this c...",c
8,Farmers in Wyoming were concerned because some...,Populations of mice and rats would increase.,[[c heres why if hawks are removed from area i...,b
9,Copper is an element that is used in electrica...,the atom,[[atom at microscopic level smallest unit of m...,a


In [71]:
accs = dftogetchal.apply(get_accc, axis = 1)
total = 0
s = 0
for i in accs:
    s = s + i[0]
    total = total + i[1]
s/total


0.4676767676767677

In [72]:
import re

def remove_non_alphanumeric_except_space(text):
    """
    Removes all characters from a string that are not alphanumeric or a space.

    Args:
        text (str): The input string.

    Returns:
        str: The string with only alphanumeric characters and spaces.
    """
    # The regex pattern `[^a-zA-Z0-9\s]+` matches one or more characters
    # that are NOT (^) lowercase letters (a-z), uppercase letters (A-Z),
    # digits (0-9), or whitespace characters (\s).
    # These matched characters are replaced with an empty string.
    cleaned_text = re.sub(r'[^a-zA-Z0-9\s]+', '', text)
    return cleaned_text

# Example usage:
input_string = "Hello, World! 123 @Python$ Program."
output_string = remove_non_alphanumeric_except_space(input_string)
print(output_string)

Hello World 123 Python Program


In [69]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from itertools import chain

def _flatten(nested):
    """Recursively flatten arbitrarily nested lists/tuples."""
    for x in nested:
        if isinstance(x, (list, tuple)):
            yield from _flatten(x)
        else:
            yield x

# Compile a few regexes to robustly catch the first answer letter near the start.
# Handles:
#   "c", "b reason...", "boxedc", "boxed c", "answer b ...", "[boxedc, c, ...]"
# We ONLY accept a/b/c/d; anything else is ignored.
_PATTERNS = [
    re.compile(r'^\s*(?:boxed)?\s*([abcd])\b', re.IGNORECASE),     # "c", "boxedc", "boxed c"
    re.compile(r'^\s*answer\s*([abcd])\b', re.IGNORECASE),         # "answer b ..."
    re.compile(r'^\s*\[?\s*(?:boxed)?\s*([abcd])\b', re.IGNORECASE) # leading bracketed tokens
]

def _extract_letter(s: str):
    """Return 'a'/'b'/'c'/'d' if a clear leading vote is present, else None."""
    if not isinstance(s, str):
        s = str(s)
    s = s.strip()
    for pat in _PATTERNS:
        m = pat.search(s)
        if m:
            letter = m.group(1).lower()
            if letter in {'a','b','c','d'}:
                return letter
    # Fallback: sometimes the very first token is just 'a'/'b'/'c'/'d' with punctuation
    m = re.match(r'^([abcd])\W', s.lower())
    if m:
        return m.group(1)
    return None

def rewrites_to_percent_matrix(df: pd.DataFrame, rewrites_col='rewrite_all_ans'):
    """
    For each row, look at all strings in `rewrites_col` (list-of-lists of strings),
    tally how many indicate A/B/C/D, and return a (n_rows x 4) NumPy array of percentages.
    Order of columns: [A, B, C, D].
    """
    out = np.zeros((len(df), 4), dtype=float)
    idx_map = {'a':0, 'b':1, 'c':2, 'd':3}

    for i, rewrites in enumerate(df[rewrites_col].tolist()):
        # Flatten any nesting and keep only strings
        flat = list(_flatten(rewrites if isinstance(rewrites, (list, tuple)) else [rewrites]))
        votes = []
        for item in flat:
            letter = _extract_letter(item)
            if letter is not None:
                votes.append(letter)

        total = len(votes)
        if total > 0:
            c = Counter(votes)
            row = np.array([
                c.get('a', 0) / total,
                c.get('b', 0) / total,
                c.get('c', 0) / total,
                c.get('d', 0) / total,
            ], dtype=float)
            out[i, :] = row
        # else: leave zeros if no recognizable votes
    return out

# --- Example usage ---
# matrix = rewrites_to_percent_matrix(df_chall, 'rewrite_all_ans')
# matrix.shape  # (n_rows, 4), columns correspond to [A, B, C, D]
# matrix[0]     # e.g. array([0.5, 0.5, 0. , 0. ])


In [70]:
def convertnums(l):
    if l == 'a':
        return 0
    elif l == 'b':
        return 1
    elif l == 'c':
        return 2
    return 3

In [22]:
df_chall_percents = rewrites_to_percent_matrix(df_chall)

NameError: name 'df_chall' is not defined

In [23]:
import numpy as np

def _bin_indices(conf, n_bins=15, strategy="uniform"):
    """
    Returns bin edges and the bin index for each confidence.
    strategy: "uniform" -> equal-width bins in [0,1]
              "quantile" -> equal-count bins (uses ranks)
    """
    conf = np.asarray(conf)
    eps = 1e-12
    if strategy == "uniform":
        edges = np.linspace(0.0, 1.0, n_bins + 1)
        # place exactly-1.0 into last bin
        idx = np.minimum(np.digitize(conf, edges, right=False) - 1, n_bins - 1)
    elif strategy == "quantile":
        # quantile edges; ensure unique and covering [0,1]
        qs = np.linspace(0, 1, n_bins + 1)
        edges = np.quantile(conf, qs)
        # handle potential duplicates (e.g., many identical confidences)
        # add tiny jitter to make edges strictly increasing
        for i in range(1, len(edges)):
            if edges[i] <= edges[i-1]:
                edges[i] = np.nextafter(edges[i-1], 1.0)
        idx = np.minimum(np.digitize(conf, edges, right=False) - 1, n_bins - 1)
        idx = np.maximum(idx, 0)
    else:
        raise ValueError("strategy must be 'uniform' or 'quantile'")
    return edges, idx

def ece_binary(y_true, y_prob, n_bins=15, strategy="uniform", return_bins=False):
    """
    ECE for binary classification.
    y_true: shape (N,), values in {0,1}
    y_prob: shape (N,), predicted probability for the positive class (class=1)
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    assert y_true.shape == y_prob.shape

    edges, idx = _bin_indices(y_prob, n_bins, strategy)
    N = len(y_true)

    ece = 0.0
    bin_stats = []
    for b in range(n_bins):
        mask = (idx == b)
        n_b = mask.sum()
        if n_b == 0:
            bin_stats.append((b, 0, np.nan, np.nan))
            continue
        acc_b = np.mean(y_true[mask] == 1)
        conf_b = np.mean(y_prob[mask])
        ece += (n_b / N) * abs(acc_b - conf_b)
        bin_stats.append((b, n_b, acc_b, conf_b))

    if return_bins:
        # returns list of tuples: (bin_id, count, accuracy, confidence)
        return float(ece), bin_stats, edges
    return float(ece)

def ece_multiclass(y_true, y_proba, n_bins=15, strategy="uniform", return_bins=False):
    """
    ECE for multiclass classification (K classes).
    y_true: shape (N,), int labels in [0, K-1]
    y_proba: shape (N, K), row i sums to 1
    Uses the standard definition with confidence = probability of the true class.
    """
    y_true = np.asarray(y_true).astype(int)
    y_proba = np.asarray(y_proba)
    N, K = y_proba.shape
    assert y_true.shape[0] == N

    # confidence for the true class
    conf_true = y_proba[np.arange(N), y_true]
    correct = (np.argmax(y_proba, axis=1) == y_true).astype(int)

    edges, idx = _bin_indices(conf_true, n_bins, strategy)

    ece = 0.0
    bin_stats = []
    for b in range(n_bins):
        mask = (idx == b)
        n_b = mask.sum()
        if n_b == 0:
            bin_stats.append((b, 0, np.nan, np.nan))
            continue
        acc_b = np.mean(correct[mask])
        conf_b = np.mean(conf_true[mask])
        ece += (n_b / N) * abs(acc_b - conf_b)
        bin_stats.append((b, n_b, acc_b, conf_b))

    if return_bins:
        return float(ece), bin_stats, edges
    return float(ece)

# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    rng = np.random.default_rng(0)
    # Binary
    y = rng.integers(0, 2, size=1000)
    p = np.clip(rng.beta(2, 5, size=1000), 0, 1)
    print("Binary ECE:", ece_binary(y, p, n_bins=15, strategy="uniform"))

    # Multiclass
    N, K = 1000, 4
    logits = rng.normal(size=(N, K))
    proba = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    y_mc = rng.integers(0, K, size=N)
    print("Multiclass ECE:", ece_multiclass(y_mc, proba, n_bins=15, strategy="quantile"))


Binary ECE: 0.2908430460849268
Multiclass ECE: 0.1991652344302426


In [24]:
df_chall_percents = rewrites_to_percent_matrix(dftogetchal)
chall_correct = [convertnums(i) for i in dftogetchal['answer_letter'].values]

NameError: name 'dftogetchal' is not defined

In [25]:
df_chall_percents

NameError: name 'df_chall_percents' is not defined

In [15]:
chall_correct

NameError: name 'chall_correct' is not defined

In [16]:
ece_multiclass(chall_correct, df_chall_percents, n_bins=5)

NameError: name 'chall_correct' is not defined

In [17]:
easy_percents = rewrites_to_percent_matrix(df_easy)
easy_correct = [convertnums(i) for i in df_easy['answer_letter'].values]

NameError: name 'rewrites_to_percent_matrix' is not defined

In [317]:
ece_multiclass(easy_correct, easy_percents, n_bins=5)

0.08384960540462452

In [90]:
with open('openbookqa.json') as train_file:
    j = json.load(train_file)

In [91]:
df_QA = pd.DataFrame(j)


In [92]:
df_QA = cleandf(df_QA)

In [93]:
df_QA

,question,answer,rewrite_all_ans,answer_letter
0,A person wants to start saving money so that t...,quit eating lunch out,[[b quit eating lunch out this choice suggests...,b
1,There is most likely going to be fog around:\n...,a marsh,[[based on general climatic conditions and geo...,a
2,Predators eat\nChoices: A. lions | B. humans |...,bunnies,[[based on general knowledge of ecosystems ans...,c
3,Oak tree seeds are planted and a sidewalk is p...,parts may break the concrete,"[[b, b, roots may be split, b, roots may be sp...",c
4,An electric car runs on electricity via\nChoic...,electrical conductors,[[c electrical conductors explanation question...,c
5,As the rain forest is deforested the atmospher...,carbon,[[oxygen deforestation or clearance of forests...,c
6,an electric car contains a motor that runs on\...,ions,[[c ions explanation electric cars are powered...,c
7,The middle of the day usually involves the bri...,human planet rotation,[[b earths planet rotation explanation at sola...,b
8,The summer solstice in the northern hemisphere...,October,"[[output may, c april, , not listed in choices...",d
9,The main component in dirt is\nChoices: A. mic...,broken stones,[[d bacteria explanation main component refers...,b


In [94]:

accs = df_QA.apply(get_accc, axis = 1)
total = 0
s = 0
for i in accs:
    s = s + i[0]
    total = total + i[1]
s/total

0.5431472081218274

In [95]:
accs

0     [49, 50]
1      [0, 50]
2     [37, 50]
3      [2, 50]
4     [28, 50]
5     [48, 50]
6     [47, 50]
7     [39, 50]
8     [21, 50]
9     [33, 50]
10     [1, 50]
11    [48, 50]
12     [4, 50]
13    [48, 50]
14    [11, 50]
15    [49, 50]
16    [50, 50]
17    [31, 40]
18    [46, 50]
19    [34, 50]
20    [37, 50]
21    [41, 50]
22    [40, 50]
23    [40, 50]
24     [3, 50]
25    [25, 50]
26     [2, 50]
27     [0, 50]
28    [36, 50]
29    [21, 40]
30    [40, 40]
31     [0, 50]
32    [48, 50]
33     [2, 50]
34     [2, 50]
35    [12, 50]
36     [1, 50]
37    [47, 50]
38    [14, 50]
39    [33, 50]
dtype: object

In [96]:
matrixqa = rewrites_to_percent_matrix(df_QA)

In [97]:
QA_correct = [convertnums(i) for i in dftogetchal['answer_letter'].values]

In [99]:
ece_multiclass(QA_correct, matrixqa, n_bins=5)

0.15635825506807238